# Глава 31. Подбор гиперпараметров

_Ноутбук сгенерирован из учебника «Машинное обучение для начинающих». Запускайте ячейки по порядку сверху вниз в Google Colab._

**Цель главы  🎯**

После этой главы ты сможешь автоматически подбирать гиперпараметры через GridSearchCV и RandomizedSearchCV, встраивать подбор в Pipeline и понимать идею современных инструментов вроде Optuna.

В главах про KNN и деревья мы подбирали K и max_depth вручную, циклом. Это работает для одного параметра, но что если их пять и у каждого десяток значений? Перебирать вручную невозможно. К счастью, sklearn умеет искать лучшие настройки автоматически.

**Аналогия: настройка гитары  💡**

Гиперпараметры — как колки на гитаре: их крутят, пока звук не станет чистым. Модель со стандартными настройками играет неплохо, но точная подстройка колков (гиперпараметров) делает звук идеальным. GridSearchCV — терпеливый мастер, который методично пробует все комбинации настроек и находит лучшее звучание. Только вместо слуха он проверяет качество по кросс-валидации.

## GridSearchCV: перебрать всё

GridSearchCV пробует все комбинации из заданной «сетки» значений и для каждой честно оценивает модель по кросс-валидации. В конце выдаёт лучшую комбинацию. Надёжно, но при многих параметрах комбинаций становится очень много.

In [ ]:
# === Подготовка: данные пингвинов ===
import pandas as pd
from sklearn.model_selection import train_test_split
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
df = pd.read_csv(url).dropna(subset=["bill_length_mm", "body_mass_g", "species"])
X = df[["bill_length_mm", "body_mass_g"]]
y = df["species"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Сетка: какие значения перебрать
grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 5, 10, None],
}

search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    grid, cv=5, scoring="accuracy")
search.fit(X_train, y_train)

print("Лучшие параметры:", search.best_params_)
print("Лучший результат:", round(search.best_score_, 3))

Здесь 3 × 4 = 12 комбинаций, каждая проверяется 5-кратной кросс-валидацией — 60 обучений модели. GridSearchCV делает это сам и запоминает лучшую.

## RandomizedSearchCV: когда комбинаций слишком много

Если параметров много, полный перебор становится неподъёмным (сотни тысяч комбинаций). RandomizedSearchCV пробует случайное подмножество — скажем, 20 случайных комбинаций — и часто находит почти такой же хороший результат в разы быстрее.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    grid, n_iter=10,      # пробуем только 10 случайных комбинаций
    cv=5, random_state=42)
search.fit(X_train, y_train)

## Подбор внутри Pipeline

Красота в том, что подбор параметров дружит с Pipeline из главы 10. Можно настраивать параметры любого шага конвейера, обращаясь к ним через имя шага и двойное подчёркивание.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

pipe = Pipeline([("sc", StandardScaler()),
                 ("knn", KNeighborsClassifier())])

# Настраиваем K внутри шага "knn" (имя шага + __ + параметр)
grid = {"knn__n_neighbors": [3, 5, 7, 9, 15]}
search = GridSearchCV(pipe, grid, cv=5).fit(X_train, y_train)
print(search.best_params_)

## Optuna — современный стандарт

**Полезно знать  ℹ️**

GridSearchCV перебирает «вслепую». Современные инструменты вроде Optuna умнее: они учатся на предыдущих попытках и направляют поиск в перспективные области, находя хорошие параметры за меньшее число проб. Это как мастер, который по первым нескольким подкруткам колков понимает, в какую сторону крутить дальше. Для больших задач Optuna — текущий стандарт; идея та же, что у GridSearchCV, но поиск умный.

**Внимание: не переусердствуй  ⚠️**

Подбор гиперпараметров обычно даёт скромный прирост (пара процентов). Хорошие признаки (глава 26) и чистые данные почти всегда важнее идеальной настройки. Не трать дни на выжимание десятых долей процента — сначала выжми максимум из данных и признаков.

## Задания

Базовый уровень

Замени ручной цикл подбора K для KNN на GridSearchCV. Найди лучшее K автоматически. Совпало ли оно с тем, что ты находил вручную в фазе 2?

Со звёздочкой  ⭐

Настрой случайный лес через GridSearchCV по двум параметрам (n_estimators, max_depth) внутри Pipeline. Сравни точность до подбора (настройки по умолчанию) и после. Насколько вырос результат? Стоил ли прирост потраченного времени?

Вопросы для обсуждения  💬

Когда стоит выбрать RandomizedSearchCV вместо полного GridSearchCV?

Почему подбор параметров оценивают по кросс-валидации, а не по одному разбиению?

Что важнее для результата — идеальная настройка параметров или хорошие признаки? Почему?

**Проверь себя  ℹ️**

1. Чем GridSearchCV отличается от RandomizedSearchCV?

2. Как настроить параметр шага внутри Pipeline?

3. Что обычно даёт больший прирост — подбор параметров или хорошие признаки?